In [2]:
import functools
import sys
from pathlib import Path
from typing import Callable

from rich import print as rprint
import math

import circuitsvis as cv
import einops
import numpy as np
import torch as t
import torch.nn as nn
import eindex
from IPython.display import display
from jaxtyping import Float, Int
from torch import Tensor
from tqdm import tqdm
from transformer_lens import (
    ActivationCache,
    FactoredMatrix,
    HookedTransformer,
    HookedTransformerConfig,
    utils,
)
from transformer_lens.hook_points import HookPoint

t.set_grad_enabled(False)

device = t.device("cuda" if t.cuda.is_available() else "mps" if t.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [3]:
cfg = HookedTransformerConfig(
  d_model=768,
  d_head=64,
  n_heads=12,
  n_layers=2,
  n_ctx=2048,
  d_vocab=50278,
  attention_dir="causal",
  attn_only=True,  # defaults to False
  tokenizer_name="EleutherAI/gpt-neox-20b",
  seed=398,
  use_attn_result=True,
  normalization_type=None,  # defaults to "LN", i.e. layernorm with weights & biases
  positional_embedding_type="shortformer",
)

In [4]:
from huggingface_hub import hf_hub_download

REPO_ID = "callummcdougall/attn_only_2L_half"
FILENAME = "attn_only_2L_half.pth"

weights_path = hf_hub_download(repo_id=REPO_ID, filename=FILENAME)

In [5]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(cfg.tokenizer_name)


In [6]:
model = HookedTransformer(cfg)
pretrained_weights = t.load(weights_path, map_location=device, weights_only=True)
model.load_state_dict(pretrained_weights)

<All keys matched successfully>

In [7]:
text = "We think that powerful, significantly superhuman machine intelligence is more likely than not to be created this century. If current machine learning techniques were scaled up to this level, we think they would by default produce systems that are deceptive or manipulative, and that no solid plans are known for how to avoid this."

logits, cache = model.run_with_cache(text, remove_batch_dim=True)

In [8]:
layer = 0
q, k = cache['q', layer], cache['k', layer]
seq, nhead, headsize = q.shape
attn_scores = einops.einsum(q, k, "q_len n_heads h_dim, k_len n_heads h_dim -> n_heads q_len k_len")
mask = t.triu(t.ones(seq, seq, dtype=t.bool), diagonal=1)
attn_scores.masked_fill_(mask, -1e9)
pattern = t.softmax(attn_scores / headsize**.5, -1)

print("Patterns match:", t.allclose(pattern, cache["pattern", layer], atol=1e-5))

Patterns match: True


In [25]:
layer = 1
tokens_ids = tokenizer.encode(text)
tokens = [tokenizer.decode(t) for t in tokens_ids]
display(
    cv.attention.attention_patterns(
        tokens=tokens,
        attention=cache["pattern", layer],
    )
)

In [24]:
n_layers = 2
def current_attn_detector(cache: ActivationCache) -> list[str]:
  """
  Returns a list e.g. ["0.2", "1.4", "1.9"] of "layer.head" which you judge to be current-token heads
  """
  threshold = 0.35
  res = []
  for layer in range(n_layers):
    seq, nhead, headsize = cache['q', layer].shape
    mask = t.eye(seq, seq).unsqueeze(0).repeat(nhead, 1, 1)
    similarity = t.sum(t.diagonal(cache['pattern', layer], offset=0, dim1=-2, dim2=-1), axis=-1) / 62
    res += [f"{layer}.{i}" for i in t.where(similarity >= threshold)[0].tolist()]

  return res

def prev_attn_detector(cache: ActivationCache) -> list[str]:
  """
  Returns a list e.g. ["0.2", "1.4", "1.9"] of "layer.head" which you judge to be prev-token heads
  """
  threshold = 0.35
  res = []
  for layer in range(n_layers):
    seq, nhead, headsize = cache['q', layer].shape
    similarity = t.sum(t.tril(t.triu(cache['pattern', layer], diagonal=-1), diagonal=-1), dim=(1,2)) / seq
    res += [f"{layer}.{i}" for i in t.where(similarity >= threshold)[0].tolist()]
  return res


def first_attn_detector(cache: ActivationCache) -> list[str]:
  """
  Returns a list e.g. ["0.2", "1.4", "1.9"] of "layer.head" which you judge to be first-token heads
  """
  threshold = 0.35
  res = []
  for layer in range(n_layers):
    similarity = t.sum(cache['pattern', layer][:, :, 0], axis=-1) / 62
    res += [f"{layer}.{i}" for i in t.where(similarity >= threshold)[0].tolist()]
  return res


print("Heads attending to current token  = ", ", ".join(current_attn_detector(cache)))
print("Heads attending to previous token = ", ", ".join(prev_attn_detector(cache)))
print("Heads attending to first token    = ", ", ".join(first_attn_detector(cache)))

Heads attending to current token  =  0.9, 0.11, 1.6
Heads attending to previous token =  0.7
Heads attending to first token    =  0.3, 1.3, 1.4, 1.8, 1.10
